# Vinyl Taste Profile & Discoveries

An analysis of my record collection — what I listen to, where it clusters, and what the algorithm has surfaced that I don't own yet.

In [1]:
import json, os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'vscode+notebook_connected'  # VS Code + HTML export
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, display

DATA_ROOT       = Path(os.environ.get('DISCOGS_DATA_ROOT', Path.home() / 'DiscogsData'))
PROFILE_PATH    = DATA_ROOT / 'profile/releases.json'
EMBEDDINGS_PATH = DATA_ROOT / 'embeddings/my_collection_embeddings.npz'
TOP_DIR         = DATA_ROOT / 'exports/top10'

profile_data = json.loads(PROFILE_PATH.read_text())
releases     = profile_data['releases']
username     = profile_data['username']

collection = [r for r in releases if r['source'] == 'collection']
wantlist   = [r for r in releases if r['source'] == 'wantlist']

print(f'User: {username}')
print(f'Collection: {len(collection)} releases  |  Wantlist: {len(wantlist)} releases')

User: barneypinkerton
Collection: 194 releases  |  Wantlist: 135 releases


---
## 1. Collection DNA

What styles, labels, and eras define the collection. Wantlist items are weighted 1.5× — they signal intent more strongly than existing purchases.

In [2]:
COLLECTION_WEIGHT = 1.0
WANTLIST_WEIGHT   = 1.5

style_raw  = defaultdict(float)
year_raw   = defaultdict(float)
label_raw  = defaultdict(float)

for r in releases:
    w = COLLECTION_WEIGHT if r['source'] == 'collection' else WANTLIST_WEIGHT
    for s in (r.get('styles') or []):
        style_raw[s] += w
    if r.get('year') and 1970 <= int(r['year']) <= 2026:
        year_raw[int(r['year'])] += w
    for l in (r.get('label_names') or []):
        label_raw[l] += w

# --- Styles ---
style_df = pd.DataFrame(
    sorted(style_raw.items(), key=lambda x: x[1], reverse=True)[:25],
    columns=['Style', 'Weight']
)
fig = px.bar(
    style_df, x='Weight', y='Style', orientation='h',
    title='Top Styles in My Collection & Wantlist',
    color='Weight', color_continuous_scale='Teal',
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False,
    plot_bgcolor='white', paper_bgcolor='white', font_color='#333333',
    title_font_size=18, height=600, margin=dict(l=140),
)
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

In [3]:
# --- Year distribution ---
year_df = pd.DataFrame(
    sorted(year_raw.items()), columns=['Year', 'Weight']
)
fig2 = px.bar(
    year_df, x='Year', y='Weight',
    title='Release Year Distribution',
    color='Weight', color_continuous_scale='Teal',
)
fig2.update_layout(
    coloraxis_showscale=False,
    plot_bgcolor='white', paper_bgcolor='white', font_color='#333333',
    title_font_size=18, height=380,
)
display(HTML(fig2.to_html(include_plotlyjs="cdn", full_html=False)))

In [4]:
# --- Top labels ---
label_df = pd.DataFrame(
    sorted(label_raw.items(), key=lambda x: x[1], reverse=True)[:20],
    columns=['Label', 'Weight']
)
fig3 = px.bar(
    label_df, x='Weight', y='Label', orientation='h',
    title='Top Labels in My Collection & Wantlist',
    color='Weight', color_continuous_scale='Teal',
)
fig3.update_layout(
    yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False,
    plot_bgcolor='white', paper_bgcolor='white', font_color='#333333',
    title_font_size=18, height=560, margin=dict(l=200),
)
display(HTML(fig3.to_html(include_plotlyjs="cdn", full_html=False)))

---
## 2. The Mean Record

Every record in the collection has an audio embedding — a 1280-dimensional fingerprint computed by Essentia's EffNet model trained on the Discogs catalogue. The centroid is the average of all those fingerprints. The track below is the one whose audio is closest to that centroid: the most *typical* sounding track in the collection.

In [5]:
# ── Paste your links for the mean record (Tripped — Wheelman) here ───────────
MEAN_RECORD_YOUTUBE_URL = 'https://www.youtube.com/watch?v=dCNY1EDwAas'  # paste YouTube URL
MEAN_RECORD_DISCOGS_URL = 'https://www.discogs.com/release/22095208-Wheelman-STE003?srsltid=AfmBOorjyL8TVtEscDB5p-ZpxKKGxjWiJV4WqAVHQQYNPC94LPjXSxy7'   # paste Discogs URL here
MEAN_RECORD_IMAGE_URL   = 'https://www.discogs.com/release/22095208-Wheelman-STE003/image/SW1hZ2U6NzMzNDE3NDg='   # paste album cover image URL here
# ──────────────────────────────────────────────────────────────────────────────

emb_data   = np.load(EMBEDDINGS_PATH)
embeddings = emb_data['embeddings'].astype(np.float32)  # (N, 1280)
centroid   = emb_data['centroid'].astype(np.float32).flatten()
filenames  = emb_data['filenames']

centroid_norm = centroid / (np.linalg.norm(centroid) + 1e-8)
norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8
sims  = (embeddings / norms) @ centroid_norm

closest_idx  = int(np.argmax(sims))
closest_sim  = float(sims[closest_idx])
closest_file = filenames[closest_idx]

stem  = Path(closest_file).stem
parts = stem.split(' - ', 2)
mean_artist  = parts[0].strip() if len(parts) >= 2 else ''
mean_release = parts[1].strip() if len(parts) >= 2 else stem
mean_track   = parts[2].strip() if len(parts) > 2 else ''

print(f'Tracks embedded:  {len(embeddings)}')
print(f'Cosine sim:       {closest_sim:.4f}')
print(f'Artist:           {mean_artist or "(unknown)"}')
print(f'Release:          {mean_release}')
print(f'Track:            {mean_track or "(full file)"}')
print(f'Raw filename:     {closest_file}')

# Build the mean record card
import re as _re
def _yt_id(url):
    m = _re.search(r'(?:v=|youtu\.be/|embed/)([\w-]{11})', url)
    return m.group(1) if m else ''

yt_id  = _yt_id(MEAN_RECORD_YOUTUBE_URL)
yt_url = MEAN_RECORD_YOUTUBE_URL.strip()
img_url = MEAN_RECORD_IMAGE_URL.strip()
disc_url = MEAN_RECORD_DISCOGS_URL.strip()

if yt_id:
    thumb = 'https://img.youtube.com/vi/' + yt_id + '/mqdefault.jpg'
    media = (
        '<a href="' + yt_url + '" target="_blank" style="display:block;position:relative;margin-bottom:12px;border-radius:8px;overflow:hidden;line-height:0">'
        '<img src="' + thumb + '" style="width:100%;height:180px;object-fit:cover;display:block">'
        '<div style="position:absolute;inset:0;display:flex;align-items:center;justify-content:center;background:rgba(0,0,0,0.35)">'
        '<div style="width:52px;height:52px;background:#dc2626;border-radius:50%;display:flex;align-items:center;justify-content:center">'
        '<div style="width:0;height:0;border-top:11px solid transparent;border-bottom:11px solid transparent;border-left:18px solid #fff;margin-left:4px"></div>'
        '</div></div></a>'
    )
elif img_url:
    media = '<img src="' + img_url + '" style="width:100%;height:180px;object-fit:cover;border-radius:8px;margin-bottom:12px;display:block">'
else:
    media = ''

discogs_btn = ('<a href="' + disc_url + '" target="_blank" style="color:#94a3b8;font-size:12px;text-decoration:none">View on Discogs →</a>') if disc_url else ''

if yt_url or img_url:
    label = 'Tripped'
    artist = 'Wheelman'
    card_html = (
        '<div style="font-family:system-ui,sans-serif;max-width:320px;margin-top:16px">'
        '<div style="background:#1e293b;border-radius:10px;padding:14px">'
        + media
        + '<div style="font-size:16px;font-weight:700;color:#f1f5f9;margin-bottom:2px">' + label + '</div>'
        + '<div style="font-size:14px;color:#94a3b8;margin-bottom:4px">' + artist + '</div>'
        + '<div style="font-size:12px;color:#64748b;margin-bottom:8px">Most typical track in collection</div>'
        + '<div style="margin-top:6px">' + discogs_btn + '</div>'
        + '</div></div>'
    )
    display(HTML(card_html))


Tracks embedded:  267
Cosine sim:       0.8658
Artist:           Wheelman
Release:          Tripped
Track:            (full file)
Raw filename:     Wheelman - Tripped.mp3


In [6]:
# Distribution of all tracks relative to centroid
sim_df = pd.DataFrame({'Cosine similarity': sims})

fig4 = px.histogram(
    sim_df, x='Cosine similarity', nbins=40,
    title='Audio Similarity to Collection Centroid — All Tracks',
    color_discrete_sequence=['#2dd4bf'],
)
fig4.add_vline(
    x=closest_sim, line_dash='dash', line_color='#f97316',
    annotation_text='← mean record', annotation_position='top right',
)
fig4.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font_color='#333333', title_font_size=18, height=380,
)
display(HTML(fig4.to_html(include_plotlyjs="cdn", full_html=False)))

---
## 3. Discoveries

Records surfaced by the recommendation pipeline that aren't in the collection. Scored by a blend of label/artist graph affinity and audio similarity to the collection centroid.

**Teal** = affinity pick (connected to known labels/artists) · **Purple** = discovery (new territory)

### Favourites so far

In [9]:
# Add any records you love here — paste the Discogs & YouTube URLs
FAVOURITES = [
    {
        'title':       'Water',
        'artists':     'Rithma',
        'labels':      '',
        'styles':      'House, Deep House',
        'country':     '',
        'year':        '',
        'note':        'A little bit odd – the standout pick so far',
        'discogs_url': 'https://www.discogs.com/release/79496',
        'youtube_url': 'https://www.youtube.com/watch?v=8jO9JuY6ZUU',
        'image_url':   'https://i.discogs.com/ZNcmeL0bnmPfSecxDYiU_58UXqP0bfkssQwmyrka7mU/rs:fit/g:sm/q:90/h:598/w:595/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTc5NDk2/LTEyMTI0Mjk3MTMu/anBlZw.jpeg',
    },
    {
        'title':       'Ping / Pong',
        'artists':     'Rob Maze',
        'labels':      '',
        'styles':      'Progressive House, House, Tech House',
        'country':     '',
        'year':        '',
        'note':        'Bouncy synths with some heavy bass',
        'discogs_url': 'https://www.discogs.com/release/72279',
        'youtube_url': 'https://www.youtube.com/watch?v=3i_ebxLAU6U',
        'image_url':   'https://i.discogs.com/nchVvcC2SHsd_2ug0uPqGJlk6OWHPB_4lmAVVyAchxE/rs:fit/g:sm/q:90/h:312/w:321/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTcyMjc5/LTExNzExMDA5ODUu/anBlZw.jpeg',
    },
    {
        'title':       'Pedro',
        'artists':     'Heads Of State',
        'labels':      '',
        'styles':      'House, Tech House',
        'country':     '',
        'year':        '',
        'note':        'Rolling groove with some trippy vocals',
        'discogs_url': 'https://www.discogs.com/release/72325',
        'youtube_url': 'https://www.youtube.com/watch?v=5zCun54rfqw',
        'image_url':   'https://i.discogs.com/tqS7do3iUmfTmUFX1trObUXEp5eN9jnsd0fZW4Vu6do/rs:fit/g:sm/q:90/h:298/w:293/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTcyMzI1/LTExMDM4MzUzMjku/anBn.jpeg',
    },
    {
        'title':       'Mobius Trip',
        'artists':     'Border Patrol (3)',
        'labels':      '',
        'styles':      'Tech House, House',
        'country':     '',
        'year':        '',
        'note':        'Something Moopie would play',
        'discogs_url': 'https://www.discogs.com/release/72467',
        'youtube_url': 'https://www.youtube.com/watch?v=xt7r_vOtIxA&t=299s',
        'image_url':   'https://i.discogs.com/bEU7YjxoRMlNDNZnbhT0sllIaqLZ3MudEZcLKyIz1R0/rs:fit/g:sm/q:90/h:600/w:600/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTcyNDY3/LTE2NzQ5MDk3ODkt/NzQwNS5qcGVn.jpeg',
    },
    {
        'title':       'Who Am I?',
        'artists':     'Rossi.',
        'labels':      '',
        'styles':      'Minimal, Dub Techno, Tech House, House',
        'country':     '',
        'year':        '',
        'note':        'Sounds like S.A.S.H breakfast',
        'discogs_url': 'https://www.discogs.com/release/20508073',
        'youtube_url': 'https://www.youtube.com/watch?v=3lzyhYleqSw',
        'image_url':   'https://i.discogs.com/DvcnjHjTU_at842GxqJtzTHmzAuWkpMOw_JW7Ewvww4/rs:fit/g:sm/q:90/h:600/w:590/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTIwNTA4/MDczLTE3NjUxNDI2/MzktMjc1Ny5qcGVn.jpeg',
    },
]


def _yt_id(url):
    import re
    m = re.search(r'(?:v=|youtu\.be/|embed/)([\w-]{11})', url)
    return m.group(1) if m else ''


def _card(r):
    meta = ' · '.join(filter(None, [r.get('labels',''), r.get('country',''), str(r.get('year',''))[:4]]))
    note = '<p style="color:#94a3b8;font-style:italic;margin:6px 0 0;font-size:13px">' + r['note'] + '</p>' if r.get('note') else ''

    yt_id  = _yt_id(r.get('youtube_url', ''))
    yt_url = r.get('youtube_url', '')
    if yt_id:
        thumb = 'https://img.youtube.com/vi/' + yt_id + '/mqdefault.jpg'
        media = (
            '<a href="' + yt_url + '" target="_blank" style="display:block;position:relative;margin-bottom:12px;border-radius:8px;overflow:hidden;line-height:0">'
            '<img src="' + thumb + '" style="width:100%;height:180px;object-fit:cover;display:block">'
            '<div style="position:absolute;inset:0;display:flex;align-items:center;justify-content:center;background:rgba(0,0,0,0.35)">'
            '<div style="width:52px;height:52px;background:#dc2626;border-radius:50%;display:flex;align-items:center;justify-content:center">'
            '<div style="width:0;height:0;border-top:11px solid transparent;border-bottom:11px solid transparent;border-left:18px solid #fff;margin-left:4px"></div>'
            '</div></div></a>'
        )
    elif r.get('image_url'):
        media = '<img src="' + r['image_url'] + '" style="width:100%;height:180px;object-fit:cover;border-radius:8px;margin-bottom:12px;display:block">'
    else:
        media = ''

    discogs_btn = (
        '<a href="' + r['discogs_url'] + '" target="_blank" style="color:#94a3b8;font-size:12px;text-decoration:none">View on Discogs →</a>'
    ) if r.get('discogs_url') else ''

    return (
        '<div style="background:#1e293b;border-radius:10px;padding:14px;margin-bottom:16px">'
        + media
        + '<div style="font-size:16px;font-weight:700;color:#f1f5f9;margin-bottom:2px">' + r['title'] + '</div>'
        + '<div style="font-size:14px;color:#94a3b8;margin-bottom:4px">' + r['artists'] + '</div>'
        + '<div style="font-size:12px;color:#64748b;margin-bottom:2px">' + r['styles'] + '</div>'
        + '<div style="font-size:12px;color:#64748b;margin-bottom:4px">' + meta + '</div>'
        + note
        + '<div style="margin-top:10px">' + discogs_btn + '</div>'
        + '</div>'
    )


html = '<div style="font-family:system-ui,sans-serif;max-width:900px;display:grid;grid-template-columns:repeat(auto-fill,minmax(280px,1fr));gap:0 16px">'
for r in FAVOURITES:
    html += _card(r)
html += '</div>'
display(HTML(html))
